## Importing Libraries

In [1]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re
import pandas as pd
import unicodedata
from groq import Groq

## Setting up files

In [10]:
GENERATION_MODEL = "deepseek-r1:8b" # qwen3:8b, llama3.1:8b, deepseek-r1:8b
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_KEY = os.getenv("GROQ_API_KEY")
CLIENT = Groq(api_key=GROQ_KEY)

TYPE_LLM = True # True - local, False - groq

DIARIES = glob.glob("../Test_Files/Clinical_diaries/inconsistancy-diary_patient_*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_diaries/Diaries-extracted/diary-extracted_patient_*.json")
NORMALIZATION_FILES = glob.glob("../Test_Files/Normalization_sheet/normalization-sheet*.csv")
SCHEMA = "../Test_Files/Schemas/parameter-extraction_schema.json"

PROMPT_FILE = "./prompts/parameter_extraction/parameter-extraction_prompt.txt"
SYS_PROMPT_FILE = "./prompts/parameter_extraction/sys_parameter-extraction_prompt.txt"

OUTPUT_DIR = "./llm-outputs/parameter-extraction/"
OUTPUT_FILE = "experiment"

NORMALIZATION_DATA = ""
sheets = {}

for file in NORMALIZATION_FILES:
    
    df_file = pd.read_csv(file, header=None)
    df_file = df_file.dropna()
    lines = df_file[0].tolist()
    
    if len(lines) < 2:
        continue
    
    group_name = lines[1].strip()
    
    terms = [line.strip() for line in lines[2:] if line.strip()]
    
    sheets[group_name] = [{"term": t} for t in terms]
    
parts = []

for group, records in sheets.items():
    parts.append(f"### {group}")
    
    for r in records:
        parts.append(f"- {r['term']}")
        
    parts.append("")
    
NORMALIZATION_DATA = "\n".join(parts)

print(f"Normalization data: {NORMALIZATION_DATA}")
print(f"Found the following diaries {DIARIES}")
print(f"Found the following golden diaries {GOLD_FILES}")
print(f"Found the following normalization files {NORMALIZATION_FILES}")

Normalization data: ### CABECA PESCOCO
- OM - CARCINOMA EPIDERMÓIDE  CAVIDADE ORAL
- OM - CARCINOMA EPIDERMÓIDE  CAVIDADE ORAL PALIATIVO
- OM - CARCINOMA EPIDERMÓIDE  CAVIDADE ORAL PALIATIVO PD-L1+
- OM - CARCINOMA  HIPOFARINGE
- OM - CARCINOMA  HIPOFARINGE PALIATIVO
- OM - CARCINOMA  HIPOFARINGE PALIATIVOPD-L1
- OM - CARCINOMA DE GLÂNDULAS SALIVARES
- OM - CARCINOMA DE GLÂNDULAS SALIVARES PALIATIVO
- OM - CARCINOMA DE GLÂNDULAS SALIVARES PALIATIVO PD-L1+
- OM - CARCINOMA FOSSAS NASAIS E SEIOS PERINANAIS
- OM - CARCINOMA FOSSAS NASAIS E SEIOS PERINANAIS PALIATIVO
- OM - CARCINOMA FOSSAS NASAIS E SEIOS PERINANAIS PALIATIVO PD-L1+
- OM - CARCINOMA LARINGE
- OM - CARCINOMA LARINGE PALIATIVO
- OM - CARCINOMA LARINGE PALIATIVO PD-L1+
- OM - CARCINOMA OROFARINGE
- OM - CARCINOMA OROFARINGE PALIATIVO
- OM - CARCINOMA OROFARINGE PALIATIVOPD-L1+
- OM - CARCINOMA EPIDERMÓIDE METASTASES CERVICAIS PRIMARIO OCULTO
- OM - CARCINOMA EPIDERMÓIDE METASTASES CERVICAIS PRIMARIO OCULTO PALIATIVO
- OM - CA

## Pre-processing

Removal of unnecessary things from the file

In [3]:
def normalize_docs(doc_content):
    text = normalize_text(doc_content)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    
    cleaned = []
    
    skip_patterns = [
        r"^UNIDADE LOCAL DE SAÚDE",
        r"^Diário Clínico$",
        r"^\d{2}-\d{2}-\d{4}",
        r"^(?:Dr|Dra|Dr\(a\))\.?\s+.*",
        r"^Processado por computador",
        r"^Pag\.\s*\d+/\d+",
    ]
    
    for line in lines:

        should_skip = any(
            re.search(pattern, line, re.IGNORECASE)
            for pattern in skip_patterns
        )
        if not should_skip:
            cleaned.append(line)
            
    text = "\n".join(cleaned)
    
    
    return text
    

def normalize_text(doc_content):
    text = unicodedata.normalize("NFKC", doc_content)
    
    text = re.sub(r"[‐-‒–—]", "-", text)

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\r\n?", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Parameter Extraction

In this phase the parameters enforced by our client will be extracted from the unstructured clinical diary through a LLM approach

In [ ]:
## Setting evironment
with open(PROMPT_FILE,"r", encoding="utf-8") as p:
    prompt_arr = [t.strip() for t in p.readlines() if t.strip()]
    base_prompt = " ".join(prompt_arr)
    
with open(SYS_PROMPT_FILE,"r", encoding="utf-8") as sp:
    sys_prompt_arr = [t.strip() for t in sp.readlines() if t.strip()]
    sys_prompt = " ".join(sys_prompt_arr)

print(f"Base prompt: {base_prompt}")
print(f"System prompt: {sys_prompt}")

os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isfile(os.path.join(OUTPUT_DIR, path)):
        count += 1
        
print(count)

In [ ]:
pbar = tqdm(total=len(DIARIES), desc="Processing diaries")

for file in DIARIES:
    patient_id = int(file.split("patient_")[-1].split(".txt")[0])
    
    with open(file,"r", encoding="utf-8") as f:

        print(f"processing file: {file}")
        
        text = f.read()
        
        normalized_text = normalize_docs(text)

        prompt_w_diary = base_prompt.replace("{{DIARY_TEXT}}", normalized_text)
        prompt = prompt_w_diary.replace("{{DIAGNOSIS_NORMALIZATION}}",NORMALIZATION_DATA)
        
        print(f"Prompt for file {file}:\n{prompt}\n")
        
        if TYPE_LLM:
            stream = chat(
                model=GENERATION_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt
                    },
                    {
                        "role": "user", 
                        "content": prompt
                        }
                    ],
                stream=True,
                options={"num_ctx": 32000}
                )
            
            llm_output = ""
            for chunk in stream:
                llm_output += chunk["message"]["content"]
                
        elif not TYPE_LLM:
            stream = CLIENT.chat.completions.create(
                model= GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": sys_prompt
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )
            
            llm_output = stream.choices[0].message.content
        

        with open(f"{OUTPUT_DIR}{OUTPUT_FILE}-{count}.txt","a",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_FILE}-{count}")
            
        
        print("\n")
    
    pbar.update(1)
        
pbar.close()

## Evaluation
In this phase the pipeline of extraction will be evaluated in 3 different fields:
- Field-level accuracy
- Missing field rate
- Schema compliance rate 

In [5]:
OUTPUT_FILES = glob.glob(f"{OUTPUT_DIR}/*-experiment-*.txt")

print(f"Found the following output files {OUTPUT_FILES}")

Found the following output files ['./llm-outputs/parameter-extraction\\deepseek-r1-8b-experiment-3.txt', './llm-outputs/parameter-extraction\\llama3.1-8b-experiment-2.txt', './llm-outputs/parameter-extraction\\qwen3-8b-experiment-1.txt']


In [24]:

for output_file in OUTPUT_FILES:
    print(f"\n\nEvaluating output file {output_file}\n\n")
    with open(output_file,"r",encoding="utf-8") as out:
        output_text = out.read()
        
        for gold_file in GOLD_FILES:
            with open(gold_file,"r",encoding="utf-8") as gf, \
                open(SCHEMA,"r",encoding="utf-8") as sch:

                curr_gf_diary = gold_file.split('_')[-2] + "_" + gold_file.split('_')[-1].split('.')[0]
                
                
                print(f"{curr_gf_diary}")
                
                data_gf = json.load(gf)
                data_sch = json.load(sch)

                outputs = output_text.split("Ouput for file ")
                outputs.pop(0)
                
                pattern = rf"diary_{re.escape(curr_gf_diary)}\b"
                
                for output in outputs:
                    if not re.search(pattern, output):
                        continue
                    
                    # Extract JSON safely
                    json_match = re.search(r"\{.*\}", output, flags=re.DOTALL)
                    if not json_match:
                        print("No JSON found for", curr_gf_diary)
                        continue

                    output_json = json.loads(json_match.group(0))

                    print("Golden truth data ", data_gf)
                    print("Output of the LLM ", output_json)

                    # Schema compliance
                    gt_keys = set(data_sch.keys())
                    out_keys = set(output_json.keys())
                    matched_keys = gt_keys & out_keys

                    print(f"The output complied with {len(matched_keys)} out of {len(gt_keys)}, "
                        f"so we have a schema compliance rate of {(len(matched_keys)/len(gt_keys))*100}%")

                    # Missing field rate
                    out_num_missing = 0
                    for key, value in data_gf.items():
                        if value is not None:
                            if key not in output_json or output_json[key] in [None, "null", ""]:
                                out_num_missing += 1

                    print(f"The output could identify {out_num_missing} fields from the golden truth diary, "
                        f"so missing field rate is {(out_num_missing/len(data_gf))*100}%")

                    # Field-level accuracy
                    
                    def normalize(x):
                        if isinstance(x, str) and x.isdigit():
                            return int(x)
                        return x

                    out_num_right = 0
                    for key in data_gf:
                        if key in output_json and normalize(data_gf[key]) == normalize(output_json[key]):
                            out_num_right += 1

                    print(f"Field-level accuracy: {out_num_right} out of {len(data_gf)} fields correctly identified, so we have a field-level accuracy of {(out_num_right/len(data_gf))*100}%")



Evaluating output file ./llm-outputs/parameter-extraction\deepseek-r1-8b-experiment-3.txt


patient_1
Golden truth data  {'age_or_birthdate': 47, 'gender': 'female', 'ecog_ps': 0, 'diagnosis': 'OM - MAMA NEOADJUVANTE TRIPLO NEGATIVO', 'diagnosis_date': None, 'molecular_status': 'ER-, PR-, HER2 0, PD-L1 CPS 15', 'stage': 'cT2N1M0 (IIIA)', 'pathology_group': 'mama', 'treatments': [{'name': 'Paclitaxel semanal + Carboplatina', 'start_date': '2025-07-01', 'end_date': '2025-11-01'}, {'name': 'AC', 'start_date': '2025-07-01', 'end_date': '2025-11-01'}], 'control': 'Resposta parcial à quimioterapia neoadjuvante; sem metástases à distância; adenopatias axilares com captação residual moderada.'}
Output of the LLM  {'age_or_birthdate': 47, 'gender': 'female', 'ecog_ps': 0, 'diagnosis': 'OM - MAMA NEOADJUVANTE TRIPLO NEGATIVO', 'diagnosis_date': None, 'molecular_status': 'triple-negative (ER-, PR-, HER2 0), PD-L1 CPS 15', 'stage': 'IIIA', 'pathology_group': 'mama', 'treatments': [{'name': 'Chem